# Algorithms for Massive Data
## Alessandro Macchi - 53633A

## Import Data

In [2]:
import os
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
# !kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows

In [3]:
import zipfile

zip_file = "imdb_data/imdb-dataset-of-top-1000-movies-and-tv-shows.zip"
extraction_path = "imdb_data"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Extracted files: {os.listdir(extraction_path)}")

Extracted files: ['imdb-dataset-of-top-1000-movies-and-tv-shows.zip', 'imdb_top_1000.csv']


## Preprocessing

In [15]:
USE_SUBSAMPLE = False
if USE_SUBSAMPLE:
    df = df.sample(frac=0.3, random_state=6)

stars_df = df[['Star1', 'Star2', 'Star3', 'Star4']]

baskets = []
for index, row in stars_df.iterrows():
    basket = [star for star in row]
    if basket:
        baskets.append(basket)

## Algorithms
### A-Priori Algorithm

In [16]:
def apriori(baskets, support_threshold):
    
    baskets = list(baskets) # otherwise it gets exhausted after the first pass

    # ==========================================
    # FIRST PASS
    # ==========================================
    names2int = {}
    id_counter = 1
    item_counts = {}

    for basket in baskets:
        for star in basket:
            if star not in names2int:
                names2int[star] = id_counter
                id_counter += 1
                item_counts[names2int[star]] = 1
            else:
                item_counts[names2int[star]] += 1

    # ==========================================
    # BETWEEN TWO PASSES
    # ==========================================
    freq_items = {}
    new_numbering = 1

    for item in item_counts:
        if item_counts[item] >= support_threshold:
            freq_items[item] = new_numbering
            new_numbering += 1
        else:
            freq_items[item] = 0

    # ==========================================
    # SECOND PASS
    # ==========================================
    pairs_freq_items = []
    for basket in baskets:
        temp_freq = []
        for item in basket:
            if freq_items[names2int[item]] != 0:
                temp_freq.append(names2int[item])

        n = len(temp_freq)
        for i in range(n):
            for j in range(i + 1, n):
                # with min/max each pair is sorted in ascending order to avoid that (A, B) =/ (B, A)
                item1 = min(temp_freq[i], temp_freq[j])
                item2 = max(temp_freq[i], temp_freq[j])

                pairs_freq_items.append((item1, item2))

        temp_freq.clear()

    counts_of_pairs = {}

    for pair in pairs_freq_items:
        if pair not in counts_of_pairs:
            counts_of_pairs[pair] = 0
        counts_of_pairs[pair] += 1

    frequent_pairs = []

    for pair, count in counts_of_pairs.items():
        if count >= support_threshold:
            actor1 = next(name for name, integer in names2int.items() if integer == pair[0]) # to go back to names
            actor2 = next(name for name, integer in names2int.items() if integer == pair[1])

            frequent_pairs.append((actor1, actor2))

    return frequent_pairs

In [17]:
apriori_stars_pairs = apriori(baskets, support_threshold=3)

### Park, Chen and Yu (PCY) Algorithm

In [18]:
def pcy(baskets, support_threshold, num_buckets=10000):

    baskets = list(baskets)

    # ==========================================
    # FIRST PASS
    # ==========================================
    names2int = {}
    id_counter = 1
    item_counts = {}
    buckets = [0] * num_buckets

    for basket in baskets:
        for star in basket:
            if star not in names2int:
                names2int[star] = id_counter
                id_counter += 1
                item_counts[names2int[star]] = 1
            else:
                item_counts[names2int[star]] += 1

        basket_ids = [names2int[star] for star in basket]
        n_items = len(basket_ids)
        for i in range(n_items):
            for j in range(i + 1, n_items):
                item1 = min(basket_ids[i], basket_ids[j])
                item2 = max(basket_ids[i], basket_ids[j])

                bucket_idx = hash((item1, item2)) % num_buckets

                buckets[bucket_idx] += 1

    # ==========================================
    # BETWEEN TWO PASSES
    # ==========================================
    freq_items = {}
    new_numbering = 1

    for item in item_counts:
        if item_counts[item] >= support_threshold:
            freq_items[item] = new_numbering
            new_numbering += 1
        else:
            freq_items[item] = 0

    bitmap = bytearray(num_buckets)

    for i in range(num_buckets):
        if buckets[i] >= support_threshold:
            bitmap[i] = 1
        else:
            bitmap[i] = 0

    del buckets #substituted by the bitmap

    # ==========================================
    # SECOND PASS
    # ==========================================
    pairs_freq_items = []
    for basket in baskets:
        temp_freq = []
        for item in basket:
            if freq_items[names2int[item]] != 0:
                temp_freq.append(names2int[item])

        n = len(temp_freq)
        for i in range(n):
            for j in range(i + 1, n):
                item1 = min(temp_freq[i], temp_freq[j])
                item2 = max(temp_freq[i], temp_freq[j])

                bucket_idx = hash((item1, item2)) % num_buckets

                if bitmap[bucket_idx] == 1:
                    pairs_freq_items.append((item1, item2))

        temp_freq.clear()

    counts_of_pairs = {}

    for pair in pairs_freq_items:
        if pair not in counts_of_pairs:
            counts_of_pairs[pair] = 0
        counts_of_pairs[pair] += 1

    frequent_pairs = []

    for pair, count in counts_of_pairs.items():
        if count >= support_threshold:
            actor1 = next(name for name, integer in names2int.items() if integer == pair[0])
            actor2 = next(name for name, integer in names2int.items() if integer == pair[1])
            frequent_pairs.append((actor1, actor2))

    return frequent_pairs

In [19]:
pcy_stars_pairs = pcy(baskets, support_threshold=3, num_buckets=len(baskets)-1)

### Savasere, Omiecinski and Navathe (SON) Algorithm

In [20]:
def son(baskets, support_threshold, num_chunks=4):

    baskets = list(baskets)
    num_baskets = len(baskets)

    # ===========================
    # Create chunks
    # ===========================
    chunk_size = num_baskets // num_chunks
    chunks = []

    for i in range(0, num_baskets, chunk_size):
        chunks.append(baskets[i:i + chunk_size])

    p = 1.0 / len(chunks)
    local_threshold = support_threshold * p

    # ==========================
    # Compute the a-priori to identify the candidate itemsets
    # ==========================
    candidate_itemsets_lists = []

    for chunk in chunks:
        candidate_itemsets_lists.append(apriori(chunk, support_threshold=local_threshold))

    unique_candidates = set()
    for local_list in candidate_itemsets_lists:
        for actor1, actor2 in local_list:
            name1 = min(actor1, actor2)
            name2 = max(actor1, actor2)
            unique_candidates.add((name1, name2))

    # ==========================
    # Count the occurences of each candidate itemset
    # ==========================

    global_counts = {}

    for basket in baskets:
        n = len(basket)
        for i in range(n):
            for j in range(i + 1, n):
                item1 = min(basket[i], basket[j])
                item2 = max(basket[i], basket[j])
                pair = (item1, item2)

                if pair in unique_candidates:
                    if pair not in global_counts:
                        global_counts[pair] = 0
                    global_counts[pair] += 1

    final_frequent_pairs = []

    for pair, count in global_counts.items():
        if count >= support_threshold:
            final_frequent_pairs.append((pair[0], pair[1]))

    return final_frequent_pairs

In [21]:
son_stars_pairs = son(baskets, support_threshold=3, num_chunks=4)

## Algorithms' Scalability

In [22]:
import time

fractions = [0.3, 0.6, 1.0]
results = []
final_insights = []

for frac in fractions:
    df_sub = df.sample(frac=frac, random_state=6)
    stars_df = df_sub[['Star1', 'Star2', 'Star3', 'Star4']]

    baskets = []
    for index, row in stars_df.iterrows():
        basket = [star for star in row if pd.notna(star)]
        if basket:
            baskets.append(basket)

    start_time = time.perf_counter()
    apriori_pairs = apriori(baskets, support_threshold=3)
    apriori_time = time.perf_counter() - start_time

    start_time = time.perf_counter()
    pcy_pairs = pcy(baskets, support_threshold=3, num_buckets=len(baskets)-1)
    pcy_time = time.perf_counter() - start_time

    start_time = time.perf_counter()
    son_pairs = son(baskets, support_threshold=3, num_chunks=4)
    son_time = time.perf_counter() - start_time

    results.append({
        'Fraction': frac,
        'Apriori Time (s)': apriori_time,
        'PCY Time (s)': pcy_time,
        'SON Time (s)': son_time
    })

results_df = pd.DataFrame(results)
print("\n--- Scalability Results ---")
display(results_df)


--- Scalability Results ---


,Fraction,Apriori Time (s),PCY Time (s),SON Time (s)
0,0.3,0.000537,0.001294,0.026789
1,0.6,0.001373,0.002976,0.082176
2,1.0,0.003090,0.010884,0.370356


In [23]:
for frac in fractions:
    def sort_pairs(pairs):
        sorted_set = set()
        for pair in pairs:
            sorted_list = sorted(pair)
            sorted_tuple = tuple(sorted_list)
            sorted_set.add(sorted_tuple)

        return sorted_set

    apriori_set = sort_pairs(apriori(baskets, support_threshold=3))
    pcy_set = sort_pairs(pcy(baskets, support_threshold=3, num_buckets=len(baskets)-1))
    son_set = sort_pairs(son(baskets, support_threshold=3, num_chunks=4))

    assert apriori_set == pcy_set, f"Mismatch between Apriori and PCY at fraction {frac}"
    assert apriori_set == son_set, f"Mismatch between Apriori and SON at fraction {frac}"
    assert pcy_set == son_set, f"Mismatch between PCY and SON at fraction {frac}"

    print("Consistency Verified: All algorithms returned the exact same itemsets.")

Consistency Verified: All algorithms returned the exact same itemsets.
Consistency Verified: All algorithms returned the exact same itemsets.
Consistency Verified: All algorithms returned the exact same itemsets.


In [24]:
# since they all give the same result we can use any set of results
print(f"Total frequent pairs found: {len(apriori_stars_pairs)}")
for pair in sorted(apriori_stars_pairs):
    print(f"- {pair[0]} & {pair[1]}")

Total frequent pairs found: 25
- Al Pacino & Diane Keaton
- Al Pacino & Robert De Niro
- Christian Bale & Michael Caine
- Daniel Radcliffe & Emma Watson
- Daniel Radcliffe & Rupert Grint
- Diane Keaton & Woody Allen
- Elijah Wood & Ian McKellen
- Elijah Wood & Orlando Bloom
- Emma Watson & Rupert Grint
- Ethan Coen & John Turturro
- Ethan Hawke & Julie Delpy
- Harrison Ford & Carrie Fisher
- Humphrey Bogart & Lauren Bacall
- Ian McKellen & Orlando Bloom
- Joe Russo & Chris Evans
- Joe Russo & Robert Downey Jr.
- Mark Hamill & Carrie Fisher
- Mark Hamill & Harrison Ford
- Robert De Niro & Joe Pesci
- Robert Downey Jr. & Chris Evans
- Robert Downey Jr. & Mark Ruffalo
- Scarlett Johansson & Chris Evans
- Tatsuya Nakadai & Toshirô Mifune
- Tom Hanks & Tim Allen
- Toshirô Mifune & Takashi Shimura
